---------------------------------------------------------------------------------
# **FINAL PRODUCTION EXPORT & HUGGING FACE PUSH (80/10/10 SPLIT)**

---------------------------------------------------------------------------------

This pipeline compiles localized token-tagging JSON datasets into a highly 
optimized cloud-native Apache Parquet dataset using Polars and PyArrow. 
After performing a deterministic 3-way split (80% Train / 10% Valid / 10% Test),
the script securely authenticates and pushes the compiled dataset directly to the 
Hugging Face Hub repository layout.

## ***Prerequisites:***
--------------------------

```cmd
    pip install polars pyarrow huggingface_hub tqdm
```

## ***Hugging Face Integration:***
-------------------------------------
Before executing, make sure you have your Hugging Face write token ready (hf_...).
You can obtain it in your Hugging Face Profile Settings -> Access Tokens.

## ***Output & Remote Target:***
-----------------------------------
    Local:  huggingface_dataset/ (train/, validation/, test/)
    Remote: https://huggingface.co/datasets/KRadim/czech-punctuation-pos-syntax

--------------------------------------------------------------------------------------

In [1]:
!apt install git-lfs




git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 138 not upgraded.


In [2]:
import os, sys
import subprocess
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

from huggingface_hub import ( 
    get_full_repo_name,
    login,
    upload_folder,
    hf_hub_download,
    HfApi
)

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("user_email")
secret_value_2 = user_secrets.get_secret("user_name")


## ***1. GLOBAL CONFIGURATION, TOKENS AND PATHS***

#### ***🔑 INSERT YOUR ACCESS KEY HERE (Must have "WRITE" permission)***

In [3]:
user_secrets = UserSecretsClient()
HF_WRITE_TOKEN = user_secrets.get_secret("HF_TOKEN")
USER_EMAIL = user_secrets.get_secret("user_email")
USER_NAME = user_secrets.get_secret("user_name")

In [4]:
def set_git_config(email, name):
    try:
        # Setting global user.email
        subprocess.run(["git", "config", "--global", "user.email", email], check=True)
        #print(f"Git user.email set to: {email}")
        
        # Setting the global user.name
        subprocess.run(["git", "config", "--global", "user.name", name], check=True)
        #print(f"Git user.name set to: {name}")
        
        # Check settings (optional)
        email_output = subprocess.run(["git", "config", "--global", "user.email"], capture_output=True, text=True, check=True)
        name_output = subprocess.run(["git", "config", "--global", "user.name"], capture_output=True, text=True, check=True)
        #print(f"Check - Email: {email_output.stdout.strip()}")
        #print(f"Check - Name: {name_output.stdout.strip()}")
        
    except subprocess.CalledProcessError as e:
        print(f"Error while setting up Git configuration: {e}")

In [5]:
set_git_config(USER_EMAIL, USER_NAME)

#### ***🏷️ HUGGING FACE REPOSITORY CONFIGURATION***

In [6]:
HF_USERNAME = "KRadim"
HF_REPO_NAME = "czech-punctuation-pos-syntax"
HF_PRIVATE_REPO = False  # Set to False if you want the dataset to be public immediately

In [7]:
INPUT_GOLD_DIR = os.path.join("/kaggle","input","notebooks","radimkzl","enrichment-czech-text-dataset","gold_datasets_output")
OUTPUT_HF_DIR = os.path.join("/kaggle","working","huggingface_dataset")

In [8]:
JSON_FILES = [
    os.path.join(INPUT_GOLD_DIR, "gold_dataset_comma.json"),
    os.path.join(INPUT_GOLD_DIR, "gold_dataset_exclamation_mark.json"),
    os.path.join(INPUT_GOLD_DIR, "gold_dataset_none.json"),
    os.path.join(INPUT_GOLD_DIR, "gold_dataset_period.json"),
    os.path.join(INPUT_GOLD_DIR, "gold_dataset_question_mark.json")
]

#### ***Strict PyArrow schema for tokens_annotation (key for Hugging Face Dataset Viewer)***

In [9]:
ARROW_SCHEMA = pa.schema([
    ("segment", pa.string()),
    ("punctuation_type", pa.string()),
    ("tokens_annotation", pa.list_(
        pa.struct([
            ("slovo", pa.string()),
            ("slovni_druh", pa.string()),
            ("vetny_clen", pa.string()),
            ("pad", pa.int64())
        ])
    ))
])

## ***2. COMPILATION, TRANSFORMATION MATRIX AND DIVISION (80 / 10 / 10)***

In [10]:
def compile_and_split_dataset():
    print("🚀 Starting compilation of the final Parquet dataset...")
    
    loaded_frames = []
    
    # 1. Step: Loading all 5 JSON files using Polars
    for file_path in JSON_FILES:
        if not os.path.exists(file_path):
            print(f"⚠️ Warning: File {file_path} not found. Skipping.")
            continue
            
        print(f"📂 Loading structured JSON: {file_path}")
        df = pl.read_json(file_path)
        
        # Fuse: If Polars interprets tokens_annotation as a pure Struct instead of List(Struct),
        # convert it to an array to fit the target PyArrow schema exactly.
        if isinstance(df.schema["tokens_annotation"], pl.Struct):
            df = df.with_columns(pl.col("tokens_annotation").implode())
            
        loaded_frames.append(df)
        
    if not loaded_frames:
        print("❌ Error: No input JSON files found in folder 'gold_datasets_output'!")
        return None

    # 2. Step: Joining all data into one DataFrame
    print("🔗 Merging all punctuation sets together...")
    combined_df = pl.concat(loaded_frames)
    total_rows = combined_df.height
    print(f"📊 Total number of rows to export: {total_rows}")

    # 3. Step: Deterministic random data shuffling
    print("🎲 Performing random row shuffling (seed=42)...")
    shuffled_df = combined_df.sample(fraction=1.0, shuffle=True, seed=42)

    # 4. Step: Calculation of indices for the distribution (80% / 10% / 10%)
    print("✂️ I divide the dataset into sections (Train: 80% | Valid: 10% | Test: 10%)...")
    train_end = int(total_rows * 0.80)
    valid_end = train_end + int(total_rows * 0.10)
    
    train_df = shuffled_df.slice(0, train_end)
    valid_df = shuffled_df.slice(train_end, valid_end - train_end)
    test_df = shuffled_df.slice(valid_end, total_rows - valid_end)
    
    print(f"  -> Train set:      {train_df.height} sentences")
    print(f"  -> Validation set: {valid_df.height} sentences")
    print(f"  -> Test set:       {test_df.height} sentences")

    # Creating an official folder structure for Hugging Face
    train_dir = os.path.join(OUTPUT_HF_DIR, "train")
    valid_dir = os.path.join(OUTPUT_HF_DIR, "validation")
    test_dir = os.path.join(OUTPUT_HF_DIR, "test")
    
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(valid_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    # 5. Step: Export to Apache Parquet via PyArrow with type checking
    print("💾 I export to Apache Parquet format with high Snappy compression...")
    
    export_jobs = [
        (train_df, train_dir, "train_data.parquet", "Train"),
        (valid_df, valid_dir, "validation_data.parquet", "Validation"),
        (test_df, test_dir, "test_data.parquet", "Test")
    ]

    for df_part, target_dir, file_name, label in export_jobs:
        output_path = os.path.join(target_dir, file_name)
        
        # Polars -> PyArrow Table -> enforce schema -> write to disk
        arrow_table = df_part.to_arrow()
        arrow_table = arrow_table.cast(ARROW_SCHEMA)
        pq.write_table(arrow_table, output_path, compression="snappy")
        print(f"    ✅ {label} set saved locally to: {output_path}")

    print("🎉 Local Parquet export completed successfully.")
    return True

## ***3. AUTOMATIZOVANÝ EXPORT NA HUGGING FACE HUB***

In [11]:
def push_to_huggingface_hub():
    print("\n🔐 Starting the upload process on Hugging Face Hub...")
    
    if HF_WRITE_TOKEN == "" or None:
        print("❌ Error: You must fill in your real HF_WRITE_TOKEN so that the script can upload data!")
        return

    # 1. API initialization and login
    try:
        login(token=HF_WRITE_TOKEN)
        api = HfApi()
        repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
        print(f"🔓 Login successful. Target repository: {repo_id}")
    except Exception as e:
        print(f"❌ Authentication to Hugging Face Hub failed: {e}")
        return

    # 2. Create a remote repository (if it doesn't already exist)
    try:
        print(f"📁 Verifying/Creating remote dataset repository...")
        api.create_repo(
            repo_id=repo_id,
            repo_type="dataset",
            private=HF_PRIVATE_REPO,
            exist_ok=True
        )
        print(f"    ✅ Repository is ready.")
    except Exception as e:
        print(f"❌ Failed to create repository on HF Hub: {e}")
        return

    # 3. Bulk and optimized upload of the entire folder (Train + Valid + Test)
    try:
        print(f"🚀 Sending the entire folder '{OUTPUT_HF_DIR}' to the cloud...")
        api.upload_folder(
            folder_path=OUTPUT_HF_DIR,
            repo_id=repo_id,
            repo_type="dataset",
            commit_message="Initial automated production parquet release (80/10/10 split)"
        )
        print("\n---------------------------------------------------------------------------")
        print(f"🔥 EVERYTHING IS DONE AND ONLINE!")
        print(f"Your dataset can be found at: https://huggingface.co/datasets/{repo_id}")
        print("---------------------------------------------------------------------------")
    except Exception as e:
        print(f"❌ Uploading data to the cloud failed: {e}")

## ***4. MAIN TRIGGER***

In [12]:
if __name__ == "__main__":
    print("📊 Research Pipeline: JSON -> Parquet (80/10/10) -> Hugging Face Dataset")
    print("---------------------------------------------------------------------------")
    
    # Step A: Starting local transformation and splitting
    success = compile_and_split_dataset()
    
    # Step B: If the local transformation passed, we will upload it to the web immediately
    if success:
        push_to_huggingface_hub()

📊 Research Pipeline: JSON -> Parquet (80/10/10) -> Hugging Face Dataset
---------------------------------------------------------------------------
🚀 Starting compilation of the final Parquet dataset...
📂 Loading structured JSON: /kaggle/input/notebooks/radimkzl/enrichment-czech-text-dataset/gold_datasets_output/gold_dataset_comma.json
📂 Loading structured JSON: /kaggle/input/notebooks/radimkzl/enrichment-czech-text-dataset/gold_datasets_output/gold_dataset_exclamation_mark.json
📂 Loading structured JSON: /kaggle/input/notebooks/radimkzl/enrichment-czech-text-dataset/gold_datasets_output/gold_dataset_none.json
📂 Loading structured JSON: /kaggle/input/notebooks/radimkzl/enrichment-czech-text-dataset/gold_datasets_output/gold_dataset_period.json
📂 Loading structured JSON: /kaggle/input/notebooks/radimkzl/enrichment-czech-text-dataset/gold_datasets_output/gold_dataset_question_mark.json
🔗 Merging all punctuation sets together...
📊 Total number of rows to export: 250000
🎲 Performing random

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


---------------------------------------------------------------------------
🔥 EVERYTHING IS DONE AND ONLINE!
Your dataset can be found at: https://huggingface.co/datasets/KRadim/czech-punctuation-pos-syntax
---------------------------------------------------------------------------
